In [1]:
import time
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, sum as spark_sum, count

# Create Spark session with better configuration for larger data
spark = SparkSession.builder \
    .appName("M4_Large_Dataset_Test") \
    .master("local[10]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "20") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Using {spark.sparkContext.defaultParallelism} cores\n")

print("=" * 60)
print("LARGER DATASET TEST: 100 million rows")
print("=" * 60)

# Create MUCH larger test data - 100 million rows
n_rows = 100_000_000
print(f"Creating {n_rows:,} rows of data...")

# For Pandas, we'll test with a smaller subset since it might run out of memory
n_rows_pandas = 20_000_000
data_pandas = {
    'id': range(n_rows_pandas),
    'value': np.random.randn(n_rows_pandas),
    'category': np.random.choice(['A', 'B', 'C', 'D', 'E'], n_rows_pandas)
}

df_pandas = pd.DataFrame(data_pandas)

print(f"\nPandas test with {n_rows_pandas:,} rows:")
start = time.time()
result_pandas = df_pandas.groupby('category')['value'].agg(['mean', 'sum', 'count'])
pandas_time = time.time() - start
print(f"Pandas time: {pandas_time:.2f} seconds")
print(result_pandas)

# For Spark, use the full 100M rows
print(f"\nPySpark test with {n_rows:,} rows:")

# Create data in chunks to avoid memory issues
chunk_size = 10_000_000
dfs = []

for i in range(n_rows // chunk_size):
    chunk_data = {
        'id': range(i * chunk_size, (i + 1) * chunk_size),
        'value': np.random.randn(chunk_size),
        'category': np.random.choice(['A', 'B', 'C', 'D', 'E'], chunk_size)
    }
    dfs.append(spark.createDataFrame(pd.DataFrame(chunk_data)))

# Union all chunks
from functools import reduce
df_spark = reduce(lambda df1, df2: df1.union(df2), dfs)

start = time.time()
result_spark = df_spark.groupBy('category').agg(
    mean('value').alias('mean'),
    spark_sum('value').alias('sum'),
    count('id').alias('count')
).toPandas()
spark_time = time.time() - start

print(f"PySpark time: {spark_time:.2f} seconds")
print(result_spark.sort_values('category'))

print(f"\nComparison:")
print(f"Pandas ({n_rows_pandas:,} rows): {pandas_time:.2f}s")
print(f"PySpark ({n_rows:,} rows): {spark_time:.2f}s")
print(f"PySpark processed {n_rows/n_rows_pandas:.0f}x more data in {spark_time/pandas_time:.2f}x the time")

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/25 11:40:59 WARN Utils: Your hostname, alpamayo.local, resolves to a loopback address: 127.0.0.1; using 10.0.1.21 instead (on interface en0)
26/01/25 11:40:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/25 11:40:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Using 10 cores

LARGER DATASET TEST: 100 million rows
Creating 100,000,000 rows of data...

Pandas test with 20,000,000 rows:
Pandas time: 0.69 seconds
              mean          sum    count
category                                
A         0.000038   153.788151  3997222
B        -0.000307 -1229.179303  4001062
C         0.000990  3961.263675  4002677
D        -0.000157  -628.342658  3999299
E        -0.000389 -1556.621420  3999740

PySpark test with 100,000,000 rows:


26/01/25 11:51:01 WARN TaskSetManager: Stage 0 contains a task of very large size (17485 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

PySpark time: 14.72 seconds
  category      mean          sum     count
1        A  0.000104  2083.293130  19997479
3        B -0.000034  -687.981300  20000078
2        C -0.000126 -2528.703566  20000145
4        D  0.000037   740.066606  20003486
0        E -0.000170 -3409.201957  19998812

Comparison:
Pandas (20,000,000 rows): 0.69s
PySpark (100,000,000 rows): 14.72s
PySpark processed 5x more data in 21.37x the time
